# Evaluation Dataset Preparation (Weak Labels + Human Gold)

This notebook prepares per-repo evaluation data in two stages:
1. scrape PR review comments and create weak labels (LLM optional)
2. merge with human annotations to produce final gold evaluation_dataset.json

Designed for team parallelization: one person per repo.

In [ ]:
# Optional install (run once)
# !pip install -q requests pandas openai

In [ ]:
from pathlib import Path
import json
import re
import time
from collections import defaultdict, Counter

import requests
import pandas as pd

ROOT = Path('..').resolve()
OUT_DIR = ROOT / 'data' / 'raw' / 'dataset_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL_DIR = OUT_DIR / 'partials_eval'
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)

REPOS = [
    'django/django',
    'pandas-dev/pandas',
    'scikit-learn/scikit-learn',
    'pallets/flask',
    'fastapi/fastapi',
]

# Choose ONE repo per person (comment/uncomment)
TARGET_REPO = 'django/django'
# TARGET_REPO = 'pandas-dev/pandas'
# TARGET_REPO = 'scikit-learn/scikit-learn'
# TARGET_REPO = 'pallets/flask'
# TARGET_REPO = 'fastapi/fastapi'

CATEGORIES = [
    'indentation',
    'naming_convention',
    'unused_import',
    'mutable_default',
    'documentation_formatting',
    'none',
]

MAX_COMMENTS = 5000
MAX_DIFF_LINES = 200
MIN_COMMENT_LEN = 15

print('Target repo:', TARGET_REPO)
print('Partial output dir:', PARTIAL_DIR)

In [ ]:
# GitHub auth
GITHUB_TOKEN = ''  # paste token
HEADERS = {'Accept': 'application/vnd.github+json'}
if GITHUB_TOKEN.strip():
    HEADERS['Authorization'] = f'Bearer {GITHUB_TOKEN.strip()}'

In [ ]:
# Weak-label model switch (comment/uncomment one)
WEAK_LABEL_MODE = 'none'  # 'none' | 'rules' | 'llm'
# WEAK_LABEL_MODE = 'rules'
# WEAK_LABEL_MODE = 'llm'

# If using LLM mode
LLM_PROVIDER = 'github_models'  # 'github_models' or 'openai'
# LLM_PROVIDER = 'openai'

LLM_MODEL = 'gpt-4.1-mini'
LLM_API_KEY = ''
LLM_BASE_URL = 'https://models.inference.ai.azure.com'  # for github_models

In [ ]:
def clean_comment(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'@[\w\-]+', '', text)
    text = re.sub(r'```[\w]*\n?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid_comment(text: str) -> bool:
    return bool(text) and len(text) >= MIN_COMMENT_LEN

KEYWORD_PATTERNS = {
    'indentation': [r'\bindent', r'\bwhitespace', r'\btab', r'\bspaces?\b', r'\balign'],
    'naming_convention': [r'\bnaming', r'\bsnake_case', r'\bcamelcase', r'\brename'],
    'unused_import': [r'\bunused import', r'\bremove import', r'\bF401\b'],
    'mutable_default': [r'\bmutable default', r'\bdefault argument', r'\bB006\b', r'\bW0102\b'],
    'documentation_formatting': [r'\bdocstring', r'\bdocumentation', r'\bPEP\s*257', r'\bcomment style'],
}

def weak_label_rules(text: str) -> str:
    t = text.lower()
    scores = {}
    for cat, pats in KEYWORD_PATTERNS.items():
        s = sum(1 for p in pats if re.search(p, t, re.IGNORECASE))
        if s > 0:
            scores[cat] = s
    if not scores:
        return 'none'
    return max(scores, key=scores.get)

def weak_label_llm(texts: list[str]) -> list[str]:
    from openai import OpenAI

    if LLM_PROVIDER == 'github_models':
        client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY or GITHUB_TOKEN)
    else:
        client = OpenAI(api_key=LLM_API_KEY)

    system = (
        'Classify each code review comment into one label: '
        'indentation, naming_convention, unused_import, mutable_default, documentation_formatting, none. '
        'Return only JSON array with one label per input.'
    )
    user = json.dumps(texts)

    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
        temperature=0.0,
        max_tokens=800,
    )

    raw = (resp.choices[0].message.content or '').strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)

    labels = json.loads(raw)
    valid = set(CATEGORIES)
    out = []
    for x in labels:
        v = str(x).strip().lower()
        out.append(v if v in valid else 'none')

    if len(out) < len(texts):
        out.extend(['none'] * (len(texts) - len(out)))
    return out[:len(texts)]

In [ ]:
def fetch_review_comments(repo: str, max_comments: int = 5000) -> list[dict]:
    owner, name = repo.split('/')
    out = []
    page = 1

    while len(out) < max_comments:
        url = f'https://api.github.com/repos/{owner}/{name}/pulls/comments?sort=created&direction=desc&per_page=100&page={page}'
        r = requests.get(url, headers=HEADERS, timeout=40)
        if r.status_code != 200:
            break

        arr = r.json()
        if not arr:
            break

        for c in arr:
            path = c.get('path') or ''
            if not path.endswith('.py'):
                continue

            body = clean_comment(c.get('body') or '')
            if not is_valid_comment(body):
                continue

            pr_url = c.get('pull_request_url') or ''
            pr_number = int(pr_url.rstrip('/').split('/')[-1]) if pr_url else None

            out.append({
                'comment_id': c.get('id'),
                'repo': repo,
                'pr_number': pr_number,
                'file_path': path,
                'line_number': c.get('line') or c.get('original_line') or 0,
                'diff_hunk': c.get('diff_hunk') or '',
                'review_comment': body,
                'created_at': c.get('created_at'),
            })

            if len(out) >= max_comments:
                break

        page += 1
        time.sleep(0.3)

    return out

comments = fetch_review_comments(TARGET_REPO, max_comments=MAX_COMMENTS)
print('Fetched comments:', len(comments))

In [ ]:
if WEAK_LABEL_MODE == 'none':
    weak = ['none'] * len(comments)
elif WEAK_LABEL_MODE == 'rules':
    weak = [weak_label_rules(c['review_comment']) for c in comments]
else:
    batch_size = 20
    weak = []
    for i in range(0, len(comments), batch_size):
        batch = comments[i:i + batch_size]
        texts = [x['review_comment'] for x in batch]
        weak.extend(weak_label_llm(texts))
        time.sleep(1.0)

for c, w in zip(comments, weak):
    c['weak_label'] = w
    c['label_source'] = 'weak_label'

print('Weak-label mode:', WEAK_LABEL_MODE)
print('Weak label counts:', dict(Counter(weak)))

In [ ]:
# Export annotation template for human labeling
slug = TARGET_REPO.replace('/', '_')
ann_path = PARTIAL_DIR / f'annotation_template_{slug}.csv'

rows = []
for c in comments:
    rows.append({
        'comment_id': c['comment_id'],
        'repo': c['repo'],
        'pr_number': c['pr_number'],
        'file_path': c['file_path'],
        'line_number': c['line_number'],
        'review_comment': c['review_comment'],
        'weak_label': c['weak_label'],
        'human_label': '',
        'annotator': '',
        'notes': '',
    })

df = pd.DataFrame(rows)
df.to_csv(ann_path, index=False, encoding='utf-8')
print('Saved annotation template:', ann_path)
print('Fill human_label with one of:', CATEGORIES)

In [ ]:
# After manual annotation is complete, set this path and run
slug = TARGET_REPO.replace('/', '_')
ANNOTATED_CSV = PARTIAL_DIR / f'annotation_template_{slug}.csv'

if not ANNOTATED_CSV.exists():
    raise FileNotFoundError(f'Annotated file not found: {ANNOTATED_CSV}')

adf = pd.read_csv(ANNOTATED_CSV)
adf['human_label'] = adf['human_label'].fillna('').astype(str).str.strip().str.lower()

valid = set(CATEGORIES)
adf = adf[adf['human_label'].isin(valid) & (adf['human_label'] != 'none')].copy()
print('Annotated usable rows:', len(adf))

grouped = defaultdict(list)
for _, r in adf.iterrows():
    key = (r['repo'], int(r['pr_number']), r['file_path'])
    grouped[key].append(r)

eval_entries = []
for (repo, pr_num, file_path), rows in grouped.items():
    chunk_lines = []
    for rr in rows:
        cc = next((c for c in comments if c['comment_id'] == rr['comment_id']), None)
        hunk = (cc.get('diff_hunk') if cc else '') or ''
        if hunk:
            chunk_lines.extend(hunk.split('\n'))

    chunk_lines = chunk_lines[:MAX_DIFF_LINES]

    gt = []
    seen = set()
    for rr in rows:
        lk = (int(rr['line_number']), rr['human_label'])
        if lk in seen:
            continue
        seen.add(lk)
        gt.append({
            'line_number': int(rr['line_number']),
            'violation_category': rr['human_label'],
            'review_comment': rr['review_comment'],
            'label_source': 'human_annotated',
            'weak_label': rr['weak_label'],
        })

    if not gt:
        continue

    eval_entries.append({
        'pr_id': f'PR_{pr_num}',
        'repo': repo,
        'file_path': file_path,
        'diff_chunks': [
            {
                'chunk_id': 'c1',
                'start_line': 0,
                'end_line': max(0, len(chunk_lines) - 1),
                'diff_lines': chunk_lines,
            }
        ],
        'ground_truth_reviews': gt,
    })

partial_eval = PARTIAL_DIR / f'eval_partial_{slug}.json'
with partial_eval.open('w', encoding='utf-8') as f:
    json.dump(eval_entries, f, indent=2, ensure_ascii=False)

print('Saved partial evaluation dataset:', partial_eval)
print('Entries:', len(eval_entries))

In [ ]:
# Merge all repo partials into final evaluation_dataset.json
partials = sorted(PARTIAL_DIR.glob('eval_partial_*.json'))
merged = []
seen = set()
for p in partials:
    with p.open('r', encoding='utf-8') as f:
        arr = json.load(f)
    for e in arr:
        k = (e['pr_id'], e['repo'], e['file_path'])
        if k in seen:
            continue
        seen.add(k)
        merged.append(e)

final_path = OUT_DIR / 'evaluation_dataset.json'
with final_path.open('w', encoding='utf-8') as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print('Merged partial files:', len(partials))
print('Final entries:', len(merged))
print('Saved:', final_path)

cat_counts = Counter(r['violation_category'] for e in merged for r in e['ground_truth_reviews'])
repo_counts = Counter(e['repo'] for e in merged)
print('Category counts:', dict(cat_counts))
print('Repo counts:', dict(repo_counts))